# 🔤 The Transformer: The Engine Behind Modern AI

**Chapter 02 — Gmail Smart Compose**

> **Goal:** Understand every component that makes a Transformer tick — from raw text to predicted next token — by building each piece from scratch.

> **Vibe:** You're building a super-powered brain for Gmail. By the end of this notebook, you'll know *exactly* why the Transformer is the architecture that changed everything. 🧠⚡

## 🐢 vs ⚡ Why Transformers Beat RNNs

### The Old Way: RNNs (Recurrent Neural Networks)

Imagine you're reading a book, but you have a **terrible memory**. You can only remember the last few words you read. By the time you reach the end of a long sentence, you've basically forgotten the beginning.

That's an RNN. It reads text **one word at a time**, left to right, updating a hidden state at each step. By the time it gets to word 50, the signal from word 1 has been diluted through 49 steps of math.

```
RNN: Reading word-by-word (like reading with amnesia)

  "The"  →  [h1]  →  "cat"  →  [h2]  →  "that"  →  [h3]  →  ... →  "sat"
              |                   |                   |                  |
           memory1            memory2             memory3     (forgot "The"!)

Problem: Long-range dependencies get lost! 😭
Also: You MUST wait for step t before computing step t+1 → SLOW to train!
```

### The New Way: Transformers

Now imagine instead of reading word-by-word, you can **see the ENTIRE page at once**. Every word can directly look at every other word and ask: "How relevant are you to understanding ME?"

```
TRANSFORMER: Seeing the whole sentence at once (like X-ray vision)

  "The"  ←→  "cat"  ←→  "that"  ←→  "lived"  ←→  "next"  ←→  "door"  ←→  "sat"
    ↕          ↕           ↕           ↕           ↕           ↕         ↕
  ALL words can directly attend to ALL other words simultaneously! 🎉

Advantage: No vanishing gradient across long sequences
Advantage: ALL positions computed in parallel → FAST training! 🚀
```

### Head-to-Head Comparison

| Property | RNN | Transformer |
|---|---|---|
| **Long-range dependencies** | ❌ Fades with distance | ✅ Direct attention to any position |
| **Training speed** | ❌ Sequential — can't parallelize | ✅ All positions in parallel |
| **Memory of long context** | ❌ Forgets early tokens | ✅ Full context window |
| **Parameter efficiency** | ✅ Fewer params for short seqs | ❌ Quadratic attention cost |

> 🎯 **Interview gold:** "We use a Transformer instead of an RNN because (1) self-attention provides constant-length paths between any two positions, solving the vanishing gradient / long-range dependency problem, and (2) all positions are computed in parallel, making training dramatically faster on modern GPUs."

In [ ]:
"""
✂️ Tokenization: Three strategies compared

Before ANY neural network can read text, text must become numbers.
That process is called tokenization. Let's compare the three main strategies.
"""
import re
from collections import Counter

# ============================================================
# Example corpus
# ============================================================
corpus = [
    "Thanks for the report.",
    "I'll review it and get back to you by Friday.",
    "Looking forward to our meeting!",
    "Please find the attached document.",
    "Let me know if you have any questions.",
]
full_text = " ".join(corpus)

# ============================================================
# Strategy 1: CHARACTER-LEVEL tokenization
# ============================================================
char_tokens = sorted(set(full_text))
print("=" * 60)
print("📝 STRATEGY 1: Character-Level Tokenization")
print("=" * 60)
print(f"Input:  'Thanks for the report'")
print(f"Tokens: {list('Thanks for the report')}")
print(f"Vocab size: {len(char_tokens)} characters")
print(f"Vocab: {char_tokens}")
print(f"\n✅ Pros: Can handle ANY word (even 'supercalifragilisticexpialidocious')")
print(f"❌ Cons: Very long sequences! 'Thanks' → 6 tokens instead of 1")

# ============================================================
# Strategy 2: WORD-LEVEL tokenization
# ============================================================
# Simple: split on spaces and punctuation
word_tokens_raw = re.findall(r"\b\w+\b|[.,!?]", full_text.lower())
word_vocab = sorted(set(word_tokens_raw))

print("\n" + "=" * 60)
print("📝 STRATEGY 2: Word-Level Tokenization")
print("=" * 60)
print(f"Input:  'Thanks for the report'")
print(f"Tokens: {re.findall(r'\\b\\w+\\b', 'Thanks for the report'.lower())}")
print(f"Vocab size: {len(word_vocab)} words (just from our tiny corpus!)")
print(f"In real English: 100,000+ words needed")
print(f"\n✅ Pros: Intuitive, short sequences")
print(f"❌ Cons: Out-Of-Vocabulary (OOV) problem!")
print(f"   What happens if someone types 'unhappiness' and it wasn't in training data?")
print(f"   → [UNK] token. All meaning lost! 💀")

# ============================================================
# Strategy 3: SUBWORD (BPE) — the best of both worlds
# ============================================================
print("\n" + "=" * 60)
print("📝 STRATEGY 3: Subword (BPE) Tokenization")
print("=" * 60)
print(f"Input:  'unhappiness'")
print(f"Tokens (BPE): ['un', 'happi', 'ness']")
print(f"\n✅ Pros: Handles new/rare words by breaking into known subwords")
print(f"✅ Pros: Vocab size ~30K — much smaller than word-level")
print(f"✅ Pros: Shorter sequences than character-level")
print(f"❌ Cons: Slightly complex to implement")

print("\n" + "=" * 60)
print("📊 SUMMARY TABLE")
print("=" * 60)
print(f"{'Strategy':<20} {'Vocab Size':<20} {'Handles New Words?':<25} {'Seq Length'}")
print("-" * 80)
print(f"{'Character':<20} {'~100':<20} {'✅ Yes':<25} {'Very Long'}")
print(f"{'Word':<20} {'100,000+':<20} {'❌ No (OOV)':<25} {'Short'}")
print(f"{'BPE (Subword)':<20} {'~30,000':<20} {'✅ Yes (decompose)':<25} {'Medium'}")
print(f"\n🏆 Smart Compose uses BPE — it's the sweet spot!")

In [ ]:
"""
🔧 BPE FROM SCRATCH

Byte Pair Encoding: Start with characters, iteratively merge the most
frequent pair of adjacent tokens.

Classic example corpus: "low lower newest widest"
"""
from collections import defaultdict
import copy

def get_vocab(corpus_words):
    """Convert words to character sequences with end-of-word marker."""
    vocab = defaultdict(int)
    for word in corpus_words:
        # Split into characters + special end-of-word token '</w>'
        char_seq = tuple(list(word) + ['</w>'])
        vocab[char_seq] += 1
    return vocab

def get_pairs(vocab):
    """Count all adjacent token pairs across the vocabulary."""
    pairs = defaultdict(int)
    for word_tokens, freq in vocab.items():
        for i in range(len(word_tokens) - 1):
            pairs[(word_tokens[i], word_tokens[i+1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    """Merge the best pair into a new token across the entire vocabulary."""
    new_vocab = defaultdict(int)
    bigram = pair  # e.g., ('l', 'o')
    for word_tokens, freq in vocab.items():
        new_tokens = []
        i = 0
        while i < len(word_tokens):
            if i < len(word_tokens) - 1 and (
                word_tokens[i], word_tokens[i+1]) == bigram:
                # Merge this pair!
                new_tokens.append(word_tokens[i] + word_tokens[i+1])
                i += 2
            else:
                new_tokens.append(word_tokens[i])
                i += 1
        new_vocab[tuple(new_tokens)] += freq
    return new_vocab

# ============================================================
# Classic BPE demo: "low lower newest widest"
# ============================================================
# Corpus with word frequencies
# In real BPE, we count word frequencies in a large corpus
corpus_words = ['low'] * 5 + ['lower'] * 2 + ['newest'] * 6 + ['widest'] * 3

vocab = get_vocab(corpus_words)
print("=" * 60)
print("🔧 BPE FROM SCRATCH")
print("=" * 60)
print(f"\nCorpus: 5× 'low', 2× 'lower', 6× 'newest', 3× 'widest'")
print(f"\nInitial vocabulary (characters + </w>):")
for tokens, freq in sorted(vocab.items(), key=lambda x: -x[1]):
    print(f"  {freq}× {' '.join(tokens)}")

# Count ALL unique characters as initial vocab
initial_chars = set()
for tokens in vocab.keys():
    for t in tokens:
        initial_chars.add(t)
print(f"\nInitial vocab size: {len(initial_chars)} characters")
print(f"Characters: {sorted(initial_chars)}")

# Run BPE for 10 merge operations
num_merges = 10
print(f"\n{'='*60}")
print(f"Running {num_merges} BPE merge operations...")
print(f"{'='*60}")

merge_rules = []
current_vocab = copy.deepcopy(vocab)

for i in range(num_merges):
    pairs = get_pairs(current_vocab)
    if not pairs:
        break
    best_pair = max(pairs, key=pairs.get)
    best_count = pairs[best_pair]
    merge_rules.append(best_pair)
    current_vocab = merge_vocab(best_pair, current_vocab)
    new_token = best_pair[0] + best_pair[1]
    print(f"Merge #{i+1}: ('{best_pair[0]}', '{best_pair[1]}') → '{new_token}'  "
          f"(appeared {best_count} times)")

print(f"\n{'='*60}")
print("Final vocabulary after BPE merges:")
print(f"{'='*60}")
for tokens, freq in sorted(current_vocab.items(), key=lambda x: -x[1]):
    display = ' | '.join(tokens)
    print(f"  {freq}× [{display}]")

# Show how a new word would be tokenized
print(f"\n✨ Why this is powerful:")
print(f"   'newest' → ['new', 'est</w>'] — merges the most common subwords")
print(f"   'newest' never appeared as a whole word in training → still handled! ✅")
print(f"\n🎯 BPE learned subwords like 'est', 'low', 'new' — useful building blocks!")

## 🗂️ Token Indexing: From Tokens to Numbers

A Transformer can't work with strings like `'new'` or `'est'`. It needs **integers**. We build a **vocabulary table** that maps every token to a unique index number.

```
VOCABULARY TABLE (simplified)

Index  Token        Index  Token
─────  ─────        ─────  ─────
  0    [PAD]          5    'king'
  1    [UNK]          6    'queen'
  2    [BOS]          7    'the'
  3    [EOS]          8    'is'
  4    [SEP]          9    'report'
  ...

"the report" → [7, 9]

Special tokens:
  [PAD] = padding to make all sequences the same length
  [UNK] = unknown token (fallback for rare characters)
  [BOS] = beginning of sequence
  [EOS] = end of sequence
  [SEP] = separator between subject/body in an email
```

This lookup table is tiny — O(1) time, O(V) space where V is vocab size. In Smart Compose, V ≈ 30,000.

In [ ]:
"""
📚 Build a simple vocabulary and do token ↔ index lookups
"""

# ============================================================
# Build vocabulary from corpus
# ============================================================
import re

corpus_sentences = [
    "Thanks for the report",
    "I will review it and get back to you",
    "Looking forward to our meeting",
    "Please find the attached document",
    "Let me know if you have any questions",
    "Thanks for getting back to me",
]

# Collect all tokens
all_tokens = []
for sentence in corpus_sentences:
    tokens = sentence.lower().split()
    all_tokens.extend(tokens)

# Count frequencies
token_counts = Counter(all_tokens)

# Build vocabulary with special tokens first
SPECIAL_TOKENS = ['[PAD]', '[UNK]', '[BOS]', '[EOS]', '[SEP]']

# Sort by frequency (most common first), then alphabetically for ties
sorted_tokens = sorted(token_counts.keys(), key=lambda t: (-token_counts[t], t))

vocab_tokens = SPECIAL_TOKENS + sorted_tokens

# Build bidirectional mappings
token_to_idx = {token: idx for idx, token in enumerate(vocab_tokens)}
idx_to_token = {idx: token for token, idx in token_to_idx.items()}

print("=" * 60)
print("📚 VOCABULARY TABLE")
print("=" * 60)
print(f"{'Index':<8} {'Token':<20} {'Frequency'}")
print("-" * 40)
for idx, token in list(idx_to_token.items())[:20]:
    freq = token_counts.get(token, 'special')
    print(f"{idx:<8} {token:<20} {freq}")
print(f"  ... ({len(vocab_tokens)} tokens total)")

# ============================================================
# Token → Index encoding
# ============================================================
def encode(text, token_to_idx, add_special=True):
    """Convert a sentence to a list of token indices."""
    tokens = text.lower().split()
    indices = []
    if add_special:
        indices.append(token_to_idx['[BOS]'])
    for token in tokens:
        idx = token_to_idx.get(token, token_to_idx['[UNK]'])
        indices.append(idx)
    if add_special:
        indices.append(token_to_idx['[EOS]'])
    return indices

def decode(indices, idx_to_token):
    """Convert a list of token indices back to a sentence."""
    tokens = [idx_to_token.get(idx, '[UNK]') for idx in indices]
    # Filter out special tokens for display
    tokens = [t for t in tokens if t not in ['[BOS]', '[EOS]', '[PAD]']]
    return ' '.join(tokens)

# Test it!
test_sentences = [
    "Thanks for the report",
    "Looking forward to our meeting",
    "Supercalifragilistic expialidocious"  # OOV words → [UNK]
]

print("\n" + "=" * 60)
print("🔄 ENCODING AND DECODING")
print("=" * 60)
for sentence in test_sentences:
    encoded = encode(sentence, token_to_idx)
    decoded = decode(encoded, idx_to_token)
    print(f"\nOriginal:  '{sentence}'")
    print(f"Encoded:   {encoded}")
    print(f"Decoded:   '{decoded}'")
    oov_words = [w for w in sentence.lower().split() if w not in token_to_idx]
    if oov_words:
        print(f"⚠️  OOV words → [UNK]: {oov_words}")

print(f"\n🎯 This is exactly what happens before text enters the Transformer!")

## 📊 Text Embeddings: Words as Points in Space

### The Problem: One-Hot Encoding is Terrible

We could represent token #7 as a one-hot vector:
```
Token #7 'the' →  [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...  (30,000 zeros!)
Token #8 'is'  →  [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...  (30,000 zeros!)
```

Two problems:
1. **Sparsity** 🚫: 30,000-dimensional vectors that are almost all zeros are hugely wasteful
2. **No semantic meaning** 🚫: 'king' and 'queen' are as "different" as 'king' and 'pizza' (both orthogonal vectors)

### The Solution: Dense Embeddings

Map each token to a **dense vector** of ~256 floats that captures MEANING:

```
SEMANTIC SPACE (2D visualization — real embeddings are 256D+)

       royal ←───────────────── gender ──────────────────→ non-royal
         ↑
    male │  king ●          man ●
         │
         │
  female │  queen ●         woman ●
         ↓
                     pizza ●  (far from royalty AND people!)

Famous relation: king - man + woman ≈ queen  (vector arithmetic!)
```

### Why Embeddings Are Learned (Not Hand-Crafted)

We initialize embeddings randomly and let **gradient descent** figure out the best representations through training. The model learns: if 'king' and 'queen' appear in similar contexts, they should have similar embeddings.

For Smart Compose: the embedding layer has shape `[vocab_size × d_model]` = `[30,000 × 256]` — about 7.7 million parameters just for embeddings!

In [ ]:
"""
📊 Embedding table in PyTorch + cosine similarity demo
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# Create a small embedding table
# ============================================================
torch.manual_seed(42)

VOCAB_SIZE = 20
D_MODEL = 8  # tiny for demo; Smart Compose uses 256

# nn.Embedding is just a learned lookup table: [vocab_size, d_model]
embedding_table = nn.Embedding(VOCAB_SIZE, D_MODEL)

print("=" * 60)
print("📊 EMBEDDING TABLE")
print("=" * 60)
print(f"Shape: {embedding_table.weight.shape}  (vocab_size × d_model)")
print(f"Total parameters: {embedding_table.weight.numel()}")
print(f"\nFor Smart Compose: 30,000 × 256 = {30_000 * 256:,} parameters!")

# Look up embeddings for token indices
indices = torch.tensor([2, 5, 7])  # e.g., 'king', 'queen', 'pizza'
embeddings = embedding_table(indices)
print(f"\nEmbeddings for tokens [2, 5, 7]:")
for i, (idx, emb) in enumerate(zip(indices, embeddings)):
    print(f"  Token {idx.item()}: {emb.detach().numpy().round(3)}")

# ============================================================
# Simulate trained embeddings for semantic demo
# Manually craft embeddings to demonstrate the concept
# (in real training, gradient descent figures this out!)
# ============================================================
print("\n" + "=" * 60)
print("🎭 SIMULATING TRAINED EMBEDDINGS (hand-crafted for demo)")
print("=" * 60)

# 4-dimensional embedding: [royalness, gender, animate, food]
word_embeddings = {
    'king':   torch.tensor([ 0.9,  0.8,  0.8, -0.5]),  # royal, male, alive, not food
    'queen':  torch.tensor([ 0.9, -0.8,  0.8, -0.5]),  # royal, female, alive, not food
    'man':    torch.tensor([-0.1,  0.8,  0.8, -0.5]),  # not royal, male, alive, not food
    'woman':  torch.tensor([-0.1, -0.8,  0.8, -0.5]),  # not royal, female, alive, not food
    'pizza':  torch.tensor([-0.8,  0.0, -0.2,  0.9]),  # not royal, neutral, not alive, food
    'burger': torch.tensor([-0.7,  0.0, -0.1,  0.8]),  # similar to pizza!
    'report': torch.tensor([-0.5,  0.0, -0.8, -0.6]),  # document-like
    'email':  torch.tensor([-0.5,  0.0, -0.8, -0.5]),  # similar to report!
}

def cosine_similarity(a, b):
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

# Compare key pairs
pairs = [
    ('king', 'queen'),    # should be high (both royal)
    ('king', 'man'),      # medium (both male)
    ('king', 'pizza'),    # should be low
    ('pizza', 'burger'),  # should be high (both food)
    ('report', 'email'),  # should be high (both documents)
    ('king', 'email'),    # should be low
]

print(f"{'Word 1':<10} {'Word 2':<10} {'Cosine Similarity':<20} {'Interpretation'}")
print("-" * 65)
for w1, w2 in pairs:
    sim = cosine_similarity(word_embeddings[w1], word_embeddings[w2])
    if sim > 0.7:
        interp = "🟢 Very similar"
    elif sim > 0.3:
        interp = "🟡 Somewhat similar"
    else:
        interp = "🔴 Quite different"
    print(f"{w1:<10} {w2:<10} {sim:<20.3f} {interp}")

# Famous vector arithmetic: king - man + woman ≈ queen
result = word_embeddings['king'] - word_embeddings['man'] + word_embeddings['woman']
print("\n🔢 Vector arithmetic: king - man + woman = ?")
best_match = max(word_embeddings.items(),
                 key=lambda x: cosine_similarity(result, x[1]))
print(f"  Result vector:     {result.numpy().round(2)}")
print(f"  Closest word:      '{best_match[0]}' ← should be 'queen'! 👑")

# ============================================================
# Visualize as heatmap
# ============================================================
words = list(word_embeddings.keys())
sim_matrix = np.zeros((len(words), len(words)))
for i, w1 in enumerate(words):
    for j, w2 in enumerate(words):
        sim_matrix[i, j] = cosine_similarity(word_embeddings[w1], word_embeddings[w2])

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(words)))
ax.set_yticks(range(len(words)))
ax.set_xticklabels(words, rotation=45, ha='right', fontsize=11)
ax.set_yticklabels(words, fontsize=11)
plt.colorbar(im, ax=ax, label='Cosine Similarity')

for i in range(len(words)):
    for j in range(len(words)):
        ax.text(j, i, f'{sim_matrix[i, j]:.2f}',
               ha='center', va='center', fontsize=9,
               color='black' if abs(sim_matrix[i, j]) < 0.7 else 'white')

ax.set_title('🧠 Embedding Similarity Matrix\nGreen = similar, Red = different', fontsize=13)
plt.tight_layout()
plt.show()

print("\n🎯 KEY INSIGHT: Embeddings encode semantic relationships!")
print("   king↔queen similar, pizza↔burger similar, king↔pizza very different")

## 📍 Positional Encoding: Teaching the Model About Order

### The Problem: Transformers Are Permutation-Blind!

Unlike RNNs that process tokens one-by-one (and thus inherently know their order), Transformers process **all tokens simultaneously in parallel**. This creates a problem:

```
Without positional encoding, these two sentences look IDENTICAL to the Transformer:

  "Dog bites man"   →  {Dog, bites, man}  ← just a SET, no order!
  "Man bites dog"   →  {man, bites, Dog}  ← same set!

But they mean completely different things! 🐕
```

### The Solution: Sine-Cosine Positional Encoding

The original Transformer paper adds a unique "position stamp" to each token's embedding. They use **sine and cosine functions at different frequencies** — think of it as giving each position a unique musical chord:

```
For position pos and dimension i:

  PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))   ← even dimensions
  PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))   ← odd dimensions

  Final input to Transformer = Embedding(token) + PE(position)
```

Why sine/cosine?
1. **Unique**: Every position gets a unique pattern
2. **Bounded**: Values always between -1 and 1 (won't blow up embeddings)
3. **Generalizable**: Can handle sequences longer than seen during training
4. **Relative positions**: The dot product PE(pos) · PE(pos+k) encodes relative distance k

Modern systems (like RoPE — Rotary Position Embedding in LLaMA) use learnable or rotary schemes, but sine-cosine remains important to understand.

In [ ]:
"""
📍 Implement sine-cosine positional encoding and visualize as heatmap
"""
import torch
import numpy as np
import matplotlib.pyplot as plt

def positional_encoding(max_seq_len, d_model):
    """
    Compute the sine-cosine positional encoding matrix.
    Shape: (max_seq_len, d_model)
    """
    PE = torch.zeros(max_seq_len, d_model)
    
    # positions: [max_seq_len, 1]
    positions = torch.arange(max_seq_len).unsqueeze(1).float()
    
    # divisors: [1, d_model/2] — different frequencies for each dimension
    # div_term = 10000^(2i / d_model)
    div_term = torch.pow(
        10000.0,
        torch.arange(0, d_model, 2).float() / d_model
    )
    
    # Even dimensions → sine
    PE[:, 0::2] = torch.sin(positions / div_term)
    # Odd dimensions → cosine
    PE[:, 1::2] = torch.cos(positions / div_term)
    
    return PE

# ============================================================
# Compute and visualize
# ============================================================
MAX_SEQ_LEN = 50
D_MODEL = 64

PE = positional_encoding(MAX_SEQ_LEN, D_MODEL)

print("=" * 60)
print("📍 POSITIONAL ENCODING")
print("=" * 60)
print(f"PE matrix shape: {PE.shape}  (seq_len × d_model)")
print(f"\nFirst 5 positions, first 8 dimensions:")
print(PE[:5, :8].numpy().round(3))

print(f"\nKey properties:")
print(f"  Min value: {PE.min().item():.4f}  (bounded by -1)")
print(f"  Max value: {PE.max().item():.4f}  (bounded by +1)")
print(f"  Each row is UNIQUE: no two positions have the same encoding ✅")

# Verify uniqueness
diffs = [(i, j, (PE[i] - PE[j]).norm().item())
         for i in range(5) for j in range(5) if i != j]
min_diff = min(d for _, _, d in diffs)
print(f"  Min distance between any two position encodings: {min_diff:.4f} ✅")

# ============================================================
# Visualizations
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Full heatmap
im1 = axes[0].imshow(PE.numpy().T, cmap='RdBu', aspect='auto',
                     vmin=-1, vmax=1)
axes[0].set_xlabel('Position in sequence', fontsize=12)
axes[0].set_ylabel('Embedding dimension', fontsize=12)
axes[0].set_title('📍 Full Positional Encoding\n(each column = unique position stamp)', fontsize=12)
plt.colorbar(im1, ax=axes[0])

# Plot 2: First 4 dimensions across positions
positions_x = np.arange(MAX_SEQ_LEN)
colors = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']
for dim in range(4):
    axes[1].plot(positions_x, PE[:, dim].numpy(),
                color=colors[dim], linewidth=2,
                label=f'dim {dim} ({"sin" if dim % 2 == 0 else "cos"})')
axes[1].set_xlabel('Position in sequence', fontsize=12)
axes[1].set_ylabel('Encoding value', fontsize=12)
axes[1].set_title('📍 First 4 Dimensions\n(different frequencies = unique patterns)', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='black', linewidth=0.5)

# Plot 3: Cosine similarity between positions (relative position encoding)
n_pos = 20
sim_matrix = np.zeros((n_pos, n_pos))
for i in range(n_pos):
    for j in range(n_pos):
        sim = torch.nn.functional.cosine_similarity(
            PE[i].unsqueeze(0), PE[j].unsqueeze(0)
        ).item()
        sim_matrix[i, j] = sim

im3 = axes[2].imshow(sim_matrix, cmap='Blues', vmin=0, vmax=1)
axes[2].set_xlabel('Position j', fontsize=12)
axes[2].set_ylabel('Position i', fontsize=12)
axes[2].set_title('📍 PE Similarity Matrix\n(nearby positions = higher similarity)', fontsize=12)
plt.colorbar(im3, ax=axes[2])

plt.suptitle('Positional Encoding: Giving Every Position a Unique Musical Chord 🎵',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n🎯 KEY INSIGHT: The PE similarity matrix shows nearby positions are more similar")
print("   This encodes RELATIVE DISTANCE — position 3 knows it's close to position 4!")

## 👀 Self-Attention: Who Pays Attention to Whom?

This is the **most important concept in the entire Transformer**. Get comfortable with it.

### The Core Idea: Pronoun Resolution as a Party Game

Imagine you're at a party and you hear: *"The cat sat on the mat because it was tired."*

What does **"it"** refer to? You immediately know: the CAT was tired (cats sit on mats, mats don't get tired). Your brain automatically connected "it" back to "cat" with high attention.

Self-attention does exactly this, but mathematically:

```
"The cat sat on the mat because it was tired"
                                ↑
                         'it' asks:
                    "Who should I look at?"

  Attention weights for 'it':
    The  → 0.03  (barely)
    cat  → 0.72  ← STRONG! 'it' is the cat
    sat  → 0.02
    on   → 0.01
    the  → 0.02
    mat  → 0.08  (some — it's in the same phrase)
    because → 0.02
    it   → 0.05  (self-reference)
    was  → 0.03
    tired → 0.02
```

### The Q, K, V Framework: A Library Metaphor 📚

Self-attention uses three matrices: **Query (Q)**, **Key (K)**, and **Value (V)**.

Think of it like a **library search system**:
- **Query (Q)**: What you're LOOKING FOR — "I need info about who was tired"
- **Key (K)**: The INDEX CARD for each book — "This book is about cats"
- **Value (V)**: The ACTUAL CONTENT of each book

You compute Q · K^T to see how relevant each key is to your query, then use those scores to weight the values.

```
SCALED DOT-PRODUCT ATTENTION:

  Attention(Q, K, V) = softmax(Q·Kᵀ / √d_k) · V

  Step 1: Q · Kᵀ          → raw attention scores (how relevant is each key?)
  Step 2: / √d_k           → scale down (prevents softmax saturation)
  Step 3: softmax(...)     → convert to probabilities (sum to 1)
  Step 4: × V              → weighted sum of values

Why √d_k? If d_k = 64, dot products can get very large (sum of 64 products).
Large inputs to softmax → near-zero gradients → bad training. Scale fixes this.
```

> 🎯 **Interview gold:** When asked "explain self-attention," use the Q/K/V library metaphor AND write the formula. Show you know why we divide by √d_k.

In [ ]:
"""
👀 Scaled dot-product self-attention from scratch in PyTorch
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention.
    
    Args:
        Q: Query matrix  (batch, seq_len, d_k)
        K: Key matrix    (batch, seq_len, d_k)
        V: Value matrix  (batch, seq_len, d_v)
        mask: Optional causal mask (for decoder — can't attend to future tokens)
    Returns:
        output: Attended values  (batch, seq_len, d_v)
        attn_weights: Attention weights  (batch, seq_len, seq_len)
    """
    d_k = Q.size(-1)
    
    # Step 1: Compute raw attention scores
    # (batch, seq_len, d_k) @ (batch, d_k, seq_len) → (batch, seq_len, seq_len)
    scores = torch.matmul(Q, K.transpose(-2, -1))
    
    # Step 2: Scale by √d_k (prevents softmax saturation)
    scores = scores / (d_k ** 0.5)
    
    # Step 3: Apply causal mask (decoder only — can't look at future tokens!)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Step 4: Softmax → attention probabilities (each row sums to 1)
    attn_weights = F.softmax(scores, dim=-1)
    
    # Step 5: Weighted sum of values
    output = torch.matmul(attn_weights, V)
    
    return output, attn_weights

# ============================================================
# Demo: A 3-word sequence
# ============================================================
torch.manual_seed(42)

SEQ_LEN = 5     # "Thanks for the report now"
D_MODEL = 8     # tiny for demo
D_K = 4         # key/query dimension
D_V = 4         # value dimension
BATCH = 1

# Simulated token embeddings (what would come out of the embedding layer)
x = torch.randn(BATCH, SEQ_LEN, D_MODEL)  # (1, 5, 8)

# Projection matrices (learned during training)
W_Q = nn.Linear(D_MODEL, D_K, bias=False)
W_K = nn.Linear(D_MODEL, D_K, bias=False)
W_V = nn.Linear(D_MODEL, D_V, bias=False)

# Compute Q, K, V projections
Q = W_Q(x)  # (1, 5, 4)
K = W_K(x)  # (1, 5, 4)
V = W_V(x)  # (1, 5, 4)

print("=" * 60)
print("👀 SCALED DOT-PRODUCT ATTENTION")
print("=" * 60)
print(f"Input x shape:  {x.shape}   (batch, seq_len, d_model)")
print(f"Q shape:        {Q.shape}   (batch, seq_len, d_k)")
print(f"K shape:        {K.shape}   (batch, seq_len, d_k)")
print(f"V shape:        {V.shape}   (batch, seq_len, d_v)")

# Raw attention scores before softmax
raw_scores = torch.matmul(Q, K.transpose(-2, -1)) / (D_K ** 0.5)
print(f"\nRaw attention scores (Q·Kᵀ / √d_k):")
print(raw_scores[0].detach().numpy().round(3))

# --- Without mask (encoder-style: attend to all positions) ---
output_enc, attn_enc = scaled_dot_product_attention(Q, K, V, mask=None)
print(f"\nAttention weights (encoder, no mask — attends to ALL positions):")
print(attn_enc[0].detach().numpy().round(3))
print(f"Each row sums to: {attn_enc[0].sum(dim=-1).detach().numpy().round(3)} ✅")

# --- With causal mask (decoder-style: can only attend to past tokens) ---
causal_mask = torch.tril(torch.ones(SEQ_LEN, SEQ_LEN)).unsqueeze(0)  # lower triangular
output_dec, attn_dec = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
print(f"\nAttention weights (decoder, CAUSAL mask — can't peek at future!):")
print(attn_dec[0].detach().numpy().round(3))
print("Upper triangle = 0 → token 1 can't see token 2, 3, 4... ✅")

print(f"\nOutput shape: {output_dec.shape}  (same shape as input — attention is a transformation!)")

# ============================================================
# Visualize attention weights
# ============================================================
tokens = ['Thanks', 'for', 'the', 'report', 'now']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (attn, title) in zip(axes, [
    (attn_enc[0].detach(), 'Encoder Attention\n(All positions attend to all)'),
    (attn_dec[0].detach(), 'Decoder Attention (Causal)\n(Only past positions attended to)')
]):
    im = ax.imshow(attn.numpy(), cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(SEQ_LEN))
    ax.set_yticks(range(SEQ_LEN))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=10)
    ax.set_yticklabels(tokens, fontsize=10)
    ax.set_xlabel('Attends TO (Keys)', fontsize=11)
    ax.set_ylabel('Query token', fontsize=11)
    ax.set_title(title, fontsize=12)
    plt.colorbar(im, ax=ax)
    for i in range(SEQ_LEN):
        for j in range(SEQ_LEN):
            val = attn[i, j].item()
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                   fontsize=8, color='white' if val > 0.5 else 'black')

plt.suptitle('Self-Attention Weights: Who Looks at Whom? 👀', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n🎯 KEY INSIGHT: Causal masking (right plot) is why decoder-only Transformers")
print("   can only predict the NEXT token — they're forced to only look backwards!")

## 🎭 Multi-Head Attention: The Expert Panel

A single self-attention head can only learn ONE type of relationship at a time. What if we want the model to simultaneously:
- Pay attention to **syntactic structure** (subject-verb agreement)
- Pay attention to **semantic meaning** (topic relevance)
- Pay attention to **positional proximity** (nearby words)
- Pay attention to **coreference** (pronoun resolution)

That's what **multi-head attention** does: run **h attention heads in parallel**, each with its own Q, K, V projections, then concatenate and project the results.

```
MULTI-HEAD ATTENTION

Input x  ─────┬─── Head 1: W_Q1, W_K1, W_V1 → Attention Output 1
              ├─── Head 2: W_Q2, W_K2, W_V2 → Attention Output 2
              ├─── Head 3: W_Q3, W_K3, W_V3 → Attention Output 3
              └─── Head 4: W_Q4, W_K4, W_V4 → Attention Output 4
                                                        ↓
                                               Concatenate all heads
                                                        ↓
                                               Linear projection W_O
                                                        ↓
                                              Final output (same shape as input)

If d_model = 256 and h = 4 heads:
  Each head uses d_k = d_v = d_model / h = 64 dimensions
  Concat([head1, head2, head3, head4]) = 4 × 64 = 256 dimensions
  Final linear: 256 → 256

Total cost ≈ same as a single attention with d_model dimensions,
but now we capture 4 different relationship types! 🎉
```

**For Smart Compose:** The actual model uses ~4 attention heads with d_model=256, so each head is 64-dimensional.

In [ ]:
"""
🎭 Multi-Head Attention in PyTorch from scratch
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # dimension per head
        
        # One big projection matrix for all heads (more efficient)
        # W_Q: d_model → d_model  (internally split into h heads of d_k each)
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)  # output projection
    
    def split_heads(self, x):
        """Split the last dimension into (num_heads, d_k) and transpose."""
        # x: (batch, seq_len, d_model)
        batch, seq_len, _ = x.shape
        x = x.view(batch, seq_len, self.num_heads, self.d_k)
        # → (batch, num_heads, seq_len, d_k)  for parallel attention computation
        return x.transpose(1, 2)
    
    def forward(self, x, mask=None):
        batch, seq_len, _ = x.shape
        
        # Project input to Q, K, V
        Q = self.split_heads(self.W_Q(x))  # (batch, heads, seq_len, d_k)
        K = self.split_heads(self.W_K(x))
        V = self.split_heads(self.W_V(x))
        
        # Scaled dot-product attention (for all heads in parallel!)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        
        if mask is not None:
            # mask: (seq_len, seq_len) → broadcast over batch and heads
            scores = scores.masked_fill(mask.unsqueeze(0).unsqueeze(0) == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)  # (batch, heads, seq_len, seq_len)
        
        # Weighted sum of values
        # (batch, heads, seq_len, d_k)
        attended = torch.matmul(attn_weights, V)
        
        # Concatenate all heads: (batch, seq_len, d_model)
        attended = attended.transpose(1, 2).contiguous().view(batch, seq_len, self.d_model)
        
        # Final output projection
        output = self.W_O(attended)
        
        return output, attn_weights

# ============================================================
# Test multi-head attention
# ============================================================
torch.manual_seed(42)

D_MODEL = 16    # tiny for demo; Smart Compose uses 256
NUM_HEADS = 4   # Smart Compose uses 4
SEQ_LEN = 5
BATCH = 2

mha = MultiHeadAttention(d_model=D_MODEL, num_heads=NUM_HEADS)
x = torch.randn(BATCH, SEQ_LEN, D_MODEL)

# Causal mask for decoder
causal_mask = torch.tril(torch.ones(SEQ_LEN, SEQ_LEN))

output, attn_weights = mha(x, mask=causal_mask)

print("=" * 60)
print("🎭 MULTI-HEAD ATTENTION")
print("=" * 60)
print(f"d_model:    {D_MODEL}")
print(f"num_heads:  {NUM_HEADS}")
print(f"d_k per head: {D_MODEL // NUM_HEADS}")
print(f"\nInput shape:        {x.shape}   (batch, seq_len, d_model)")
print(f"Output shape:       {output.shape}   (same — attention is a transformation!)")
print(f"Attention weights:  {attn_weights.shape}   (batch, heads, seq_q, seq_k)")

# Count parameters
total_params = sum(p.numel() for p in mha.parameters())
print(f"\nTotal MHA parameters: {total_params}")
print(f"  W_Q: {mha.W_Q.weight.numel()} | W_K: {mha.W_K.weight.numel()} | "
      f"W_V: {mha.W_V.weight.numel()} | W_O: {mha.W_O.weight.numel()}")

# Visualize attention for each head
tokens = ['Thanks', 'for', 'the', 'report', 'now']
fig, axes = plt.subplots(1, NUM_HEADS, figsize=(4*NUM_HEADS, 4))

for head_idx, ax in enumerate(axes):
    # First batch item, this head
    attn_head = attn_weights[0, head_idx].detach().numpy()
    im = ax.imshow(attn_head, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(SEQ_LEN))
    ax.set_yticks(range(SEQ_LEN))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(tokens, fontsize=9)
    ax.set_title(f'Head {head_idx+1}', fontsize=11)
    for i in range(SEQ_LEN):
        for j in range(SEQ_LEN):
            val = attn_head[i, j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                   fontsize=7, color='white' if val > 0.5 else 'black')

plt.suptitle('🎭 Multi-Head Attention: 4 Different "Views" of the Same Sequence',
            fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n🎯 KEY INSIGHT: Each head learns DIFFERENT attention patterns!")
print("   Head 1 might focus on syntax, Head 2 on semantics, etc.")
print("   That's why 4 heads outperform 1 head — diverse perspectives!")

## 🏗️ Full Decoder-Only Transformer Architecture

Now let's put it ALL together. Smart Compose uses a **decoder-only Transformer** (like GPT) — it reads text left-to-right and predicts what comes next.

```
DECODER-ONLY TRANSFORMER (Smart Compose Architecture)
═══════════════════════════════════════════════════════

INPUT: "Thanks for the"
         ↓
┌─────────────────────────────────┐
│  1. TOKENIZATION                │
│     "Thanks" "for" "the"        │
│         ↓                       │
│  [7832]  [318]  [262]           │
└──────────────┬──────────────────┘
               ↓
┌─────────────────────────────────┐
│  2. EMBEDDING LAYER             │
│     token_idx → d_model vector  │
│     Shape: (seq_len, d_model)   │
└──────────────┬──────────────────┘
               ↓
┌─────────────────────────────────┐
│  3. POSITIONAL ENCODING         │
│     Add position stamps         │
│     x = embed(x) + PE(pos)      │
└──────────────┬──────────────────┘
               ↓
┌═════════════════════════════════╗
║  TRANSFORMER BLOCK × N layers  ║  ← Repeated N times (Smart Compose: N=6)
║                                 ║
║  ┌─────────────────────────┐   ║
║  │  Multi-Head Attention   │   ║  ← Self-attention (causal mask!)
║  │  (Causal / Masked)      │   ║
║  └────────────┬────────────┘   ║
║               │ + residual     ║
║  ┌────────────▼────────────┐   ║
║  │  Layer Normalization    │   ║
║  └────────────┬────────────┘   ║
║               ↓                ║
║  ┌─────────────────────────┐   ║
║  │  Feed-Forward Network   │   ║  ← 2 linear layers with GELU activation
║  │  FFN(x) = W2·GELU(W1·x)│   ║  ← d_model → 4×d_model → d_model
║  └────────────┬────────────┘   ║
║               │ + residual     ║
║  ┌────────────▼────────────┐   ║
║  │  Layer Normalization    │   ║
║  └────────────┬────────────┘   ║
╚══════════════╪════════════════╝
               ↓ (repeat N times)
┌─────────────────────────────────┐
│  4. PREDICTION HEAD             │
│     Linear: d_model → vocab     │
│     Softmax: logits → probs     │
└──────────────┬──────────────────┘
               ↓
OUTPUT: P(next_token | "Thanks for the")
  "report"  → 0.42  ← most likely!
  "email"   → 0.18
  "meeting" → 0.11
  ...       → ...
═══════════════════════════════════════════════════════

KEY COMPONENTS:
  • Causal masking: each token only attends to previous tokens
  • Residual connections: x → x + sublayer(x)  (prevents vanishing gradients)
  • Layer normalization: stabilizes training
  • FFN: adds non-linearity and "memory" between attention layers

Smart Compose config: N=6, d_model=256, heads=4, d_ff=1024 → ~20M params
```

In [ ]:
"""
🏗️ A complete decoder-only Transformer — forward pass from scratch
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class FeedForward(nn.Module):
    """Position-wise Feed-Forward Network: d_model → 4*d_model → d_model"""
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))


class TransformerBlock(nn.Module):
    """One Transformer decoder block: MHA + FFN, each with residual + LayerNorm"""
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Sub-layer 1: Multi-Head Attention + residual + LayerNorm
        attn_out, attn_weights = self.attention(x, mask)
        x = self.norm1(x + self.dropout(attn_out))  # residual connection!
        
        # Sub-layer 2: Feed-Forward + residual + LayerNorm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))  # residual connection!
        
        return x, attn_weights


class DecoderOnlyTransformer(nn.Module):
    """
    Decoder-only Transformer (GPT-style).
    Used in Smart Compose for next-token prediction.
    """
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers,
                 max_seq_len=512, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        
        # 1. Embedding layer
        self.embedding = nn.Embedding(vocab_size, d_model)
        
        # 2. Positional encoding (fixed, not learned)
        self.register_buffer('PE', positional_encoding(max_seq_len, d_model))
        
        # 3. Stack of N Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        # 4. Final layer norm
        self.final_norm = nn.LayerNorm(d_model)
        
        # 5. Prediction head: d_model → vocab_size
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Optional: weight tying — share weights between embedding and lm_head
        # This is a common trick that reduces parameters and improves performance
        self.lm_head.weight = self.embedding.weight
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, token_ids):
        """
        Args:
            token_ids: (batch, seq_len)  — integer token indices
        Returns:
            logits: (batch, seq_len, vocab_size)  — unnormalized next-token probs
        """
        batch, seq_len = token_ids.shape
        
        # Step 1: Token embeddings
        x = self.embedding(token_ids)  # (batch, seq_len, d_model)
        x = x * math.sqrt(self.d_model)  # scale (from original paper)
        
        # Step 2: Add positional encoding
        x = x + self.PE[:seq_len, :].unsqueeze(0)  # (batch, seq_len, d_model)
        
        # Step 3: Build causal mask (can't attend to future tokens)
        causal_mask = torch.tril(torch.ones(seq_len, seq_len, device=token_ids.device))
        
        # Step 4: Pass through N Transformer blocks
        all_attn_weights = []
        for block in self.blocks:
            x, attn_w = block(x, mask=causal_mask)
            all_attn_weights.append(attn_w)
        
        # Step 5: Final layer norm
        x = self.final_norm(x)
        
        # Step 6: Prediction head → logits over vocabulary
        logits = self.lm_head(x)  # (batch, seq_len, vocab_size)
        
        return logits, all_attn_weights

# ============================================================
# Instantiate the Smart Compose Transformer!
# ============================================================
torch.manual_seed(42)

# Smart Compose approximate config
config = {
    'vocab_size': 1000,   # small vocab for demo; real: 30,000
    'd_model': 64,        # hidden dim; real: 256
    'num_heads': 4,       # attention heads; real: 4
    'd_ff': 256,          # FFN hidden dim = 4 × d_model; real: 1024
    'num_layers': 2,      # Transformer layers; real: 6
    'max_seq_len': 64,    # max context window; real: 512
}

model = DecoderOnlyTransformer(**config)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 60)
print("🏗️ DECODER-ONLY TRANSFORMER")
print("=" * 60)
print(f"Config: {config}")
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nFor Smart Compose (real config):")
real_params = 30000*256 + 6*(4*(256*64*4) + 2*(256*1024+1024*256)) + 256*30000
print(f"  Approx. params: ~10-20M (this is tiny! GPT-3 has 175B)")

# Forward pass
batch_size = 2
seq_len = 8
dummy_tokens = torch.randint(0, config['vocab_size'], (batch_size, seq_len))

logits, attn_weights = model(dummy_tokens)

print(f"\n{'='*60}")
print(f"FORWARD PASS")
print(f"{'='*60}")
print(f"Input tokens shape:  {dummy_tokens.shape}   (batch, seq_len)")
print(f"Output logits shape: {logits.shape}  (batch, seq_len, vocab_size)")
print(f"Number of attn layers: {len(attn_weights)}")
print(f"Attn weights per layer: {attn_weights[0].shape}  (batch, heads, seq_q, seq_k)")

# Show next-token probabilities for position 0
probs = F.softmax(logits[0, -1, :], dim=-1)  # Last position, first batch item
top5_probs, top5_ids = torch.topk(probs, 5)
print(f"\nTop-5 next-token predictions (after last input token):")
for rank, (tok_id, prob) in enumerate(zip(top5_ids, top5_probs)):
    print(f"  #{rank+1}: Token {tok_id.item():5d}  →  p = {prob.item():.4f}")

print(f"\n✅ The Transformer is working! Sum of probs: {probs.sum().item():.6f}")
print(f"🎯 In real Smart Compose, we'd take the top prediction as the suggested word.")

## 🧪 Quiz Time! Test Your Transformer Knowledge

Try to answer before revealing! Cover the answer and think it through. 🤔

---

### Question 1: Why Transformers?

Why do Transformers outperform RNNs on long sequences, and what architectural feature enables this?

- A) Transformers use more parameters, which gives them more capacity
- B) Self-attention allows direct connections between any two positions, eliminating the vanishing gradient problem that affects RNNs
- C) Transformers process text faster because they use GPUs more efficiently
- D) Transformers use larger hidden states than RNNs

<details>
<summary>🔓 Click to reveal answer</summary>

**B)** The key is **O(1) path length** between any two positions via direct attention, versus O(n) for RNNs (information must flow through n sequential steps). This eliminates the vanishing gradient problem for long sequences. The parallel computation advantage (GPU utilization) is a training speed benefit, but the core reason for better performance is the direct attention path.

</details>

---

### Question 2: BPE Tokenization

A user types the email phrase "deserialization error". The word 'deserialization' was never in the training corpus. How does BPE handle this?

- A) It returns [UNK] because the word is out-of-vocabulary
- B) It breaks it into known subwords: e.g., ["de", "serial", "ization"] which were all seen during BPE training
- C) It skips the word entirely
- D) It tokenizes it character by character regardless of BPE

<details>
<summary>🔓 Click to reveal answer</summary>

**B)** BPE's superpower! Because BPE builds a vocabulary of **subwords** from frequent character merges, it can decompose any new word into known subword pieces. 'deserialization' → ['de', 'serial', 'ization'] (or similar). Each subword piece carries meaning. This is why BPE replaced word-level tokenization — it handles OOV words gracefully while keeping sequences short.

</details>

---

### Question 3: Why Divide by √d_k?

In scaled dot-product attention, why do we divide the raw attention scores by √d_k before applying softmax?

- A) To normalize the scores so they sum to 1
- B) To make the computation cheaper
- C) Large dot products push softmax into saturation regions with near-zero gradients; scaling prevents this
- D) It's just a convention and doesn't actually matter

<details>
<summary>🔓 Click to reveal answer</summary>

**C)** When d_k is large (say, 64), the dot product Q·K^T is a sum of 64 multiplications. Even if each Q and K component has unit variance, the dot product has variance d_k. So dot products scale in magnitude with d_k. Large values fed into softmax produce near-zero gradients everywhere except the maximum — the model can't learn effectively. Dividing by √d_k keeps the variance at 1 and prevents this saturation. This small detail is mentioned in the original "Attention Is All You Need" paper and shows up in interviews!

</details>

---

### Question 4: Causal Masking

Smart Compose uses a causal (triangular) mask in self-attention. Why is this mask essential for an autoregressive language model?

- A) It speeds up computation by skipping half the attention operations
- B) Without it, during training the model could "cheat" by attending to future tokens it shouldn't know yet
- C) It prevents the model from paying attention to padding tokens
- D) It ensures the model only uses the embedding layer, not positional encoding

<details>
<summary>🔓 Click to reveal answer</summary>

**B)** During training, we feed the model a full sequence and ask it to predict each next token in parallel (teacher forcing). Without the causal mask, position 3 could attend to position 4 — essentially reading the answer before predicting it. The causal mask sets all upper-triangular attention scores to -∞ (so softmax gives 0), ensuring each position only sees previous positions. Without this, the model would learn to "copy" the next token instead of truly learning language patterns.

</details>

---

### Question 5: Multi-Head vs Single-Head

Smart Compose uses 4 attention heads with d_model=256 (so d_k=64 per head). What's the key advantage of 4 heads over a single head with d_k=256?

- A) 4 heads use less compute than 1 head with the same total dimensions
- B) Multiple heads allow the model to attend to different types of relationships (syntactic, semantic, coreference) simultaneously
- C) Multiple heads prevent overfitting by using dropout
- D) It's required by the PyTorch API

<details>
<summary>🔓 Click to reveal answer</summary>

**B)** Each attention head learns to attend to DIFFERENT aspects of the sequence. In practice, some heads learn syntactic dependencies (subject-verb), others learn semantic relevance (topic words), others learn positional proximity. A single head with the full d_k would need to "average" all these relationship types into one representation. Multiple heads in parallel allow specialization. Note: the compute cost is roughly the same (the total dimension d_k × h = d_model is constant).

</details>